# Thematische Analyse
Kongruenz nach Thema, Abstimmungstyp und Zeit.

In [ ]:
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2
from visualisierungen import *

df = pd.read_csv("../data/processed/df_with_positions.csv")
print(df.info(50))

## 3. Kongruenz nach Thema (`hauptgruppe`)

In [ ]:
# Alle Zustimmungs-Spalten sammeln
boxplot(df, x="hauptgruppe", y="zustimmung_br-pos",
        titel="Zustimmung nach Hauptgruppe",
        xlabel="Hauptgruppe", ylabel="Zustimmung",
        figsize=(10, 5), rotation=20)

In [ ]:
bund_cols = ["zustimmung_br-pos", "zustimmung_p-svp", "zustimmung_p-fdp", "zustimmung_p-mitte",
            "zustimmung_p-sps", "zustimmung_p-gps", "zustimmung_p-glp"]

df_long = df[["hauptgruppe"] + bund_cols].melt(
    id_vars="hauptgruppe", var_name="bund", value_name="zustimmung")

df_long['bund'] = df_long['bund'].str.replace('zustimmung_', '')

boxplot(df_long, x="hauptgruppe", y="zustimmung", hue="bund", palette = PALETTE_KATEGORIAL_VIELE_WERTE,
        titel="Zustimmung Bundesebene",
        xlabel="Bund", ylabel="Zustimmung", hline = 0, width = 0.8,
        figsize=(30, 20))

In [ ]:
df_long = df[["jahrzehnt", "hauptgruppe"] + bund_cols].melt(
    id_vars=["jahrzehnt", "hauptgruppe"], var_name="bund", value_name="zustimmung")

df_long['bund'] = df_long['bund'].str.replace('zustimmung_', '')

scatterplot(df_long, x="jahrzehnt", y="zustimmung", hue="bund", palette = PALETTE_KATEGORIAL_VIELE_WERTE,
        titel="Zustimmung Bundesebene",
        xlabel="Bund", ylabel="Zustimmung",
        figsize=(30, 20))

In [ ]:
pivot = df_long.groupby(["hauptgruppe", "bund"])["zustimmung"].mean().unstack()

heatmap(pivot, xlabel="Akteur", ylabel="Hauptgruppe",
        cmap=CMAP_DIVERGIEREND, fmt=".2f", figsize=(12, 8), rotation=0)

In [ ]:
plt.close('all')

hauptgruppen = sorted(df_long['hauptgruppe'].dropna().unique())
n_groups = len(hauptgruppen)

fig, axes = plt.subplots(n_groups, 1, figsize=(12, 5 * n_groups), sharex=True, sharey=True)

for ax, gruppe in zip(axes, hauptgruppen):
    subset = df_long[df_long['hauptgruppe'] == gruppe]

    # Mittelwert wie gehabt
    means = subset.groupby(["jahrzehnt", "bund"])["zustimmung"].mean().reset_index()

    # Anzahl separat berechnen und mergen
    counts = subset.groupby(["jahrzehnt", "bund"]).size().reset_index(name='n')
    means = means.merge(counts, on=["jahrzehnt", "bund"])

    sns.lineplot(data=means, x="jahrzehnt", y="zustimmung", hue="bund",
                 marker="o", palette=PALETTE_KATEGORIAL_VIELE_WERTE[:4], ax=ax)

    # Wert + n an jedem Punkt
    for _, row in means.iterrows():
        ax.annotate(f"{row['zustimmung']:.1f}\n(n={row['n']})",
                    xy=(row['jahrzehnt'], row['zustimmung']),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=7, color='#444444')

    # Gesamt-N pro Jahrzehnt (einmal oben)
    n_pro_jahrzehnt = subset.groupby('jahrzehnt').size().reset_index(name='n_total')
    y_pos = means['zustimmung'].max() + 0.15

    for _, row in n_pro_jahrzehnt.iterrows():
        ax.annotate(f"N={row['n_total']}",
                    xy=(row['jahrzehnt'], y_pos),
                    ha='center', fontsize=9, color='#222222',
                    fontweight='bold')

    ax.axhline(0, color="#888888", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title(gruppe, fontsize=14)
    ax.set_ylabel("Zustimmung")
    ax.tick_params(axis='x', labelbottom=True, rotation=45)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

axes[-1].set_xlabel("Jahrzehnt")
plt.tight_layout()
plt.show()

In [ ]:
zust_cols = [c for c in df.columns if c.startswith("zustimmung_p-")]
df_long = df.melt(id_vars=["hauptgruppe"], value_vars=zust_cols,
                   var_name="partei", value_name="zustimmung").dropna()
df_long["partei"] = df_long["partei"].str.replace("zustimmung_", "")

parteien_auswahl = ["p-svp", "p-fdp", "p-mitte", "p-sps", "p-gps", "p-glp"]
sub = df_long[df_long["partei"].isin(parteien_auswahl)]
hgs = sub["hauptgruppe"].unique()

fig, axes = plt.subplots(4, 3, figsize=(16, 14), sharex=True)
axes = axes.flatten()

for i, hg in enumerate(hgs):
    ax = axes[i]
    s = sub[sub["hauptgruppe"] == hg]
    stats = s.groupby("partei")["zustimmung"].agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
    stats.columns = ["median", "q25", "q75"]
    stats = stats.loc[stats.index.isin(parteien_auswahl)].sort_values("median")

    for j, (p, row) in enumerate(stats.iterrows()):
        ax.errorbar(row["median"], j,
                     xerr=[[row["median"]-row["q25"]], [row["q75"]-row["median"]]],
                     fmt="o", capsize=4, markersize=6)
    ax.axvline(0, color="grey", ls="--", alpha=0.5)
    ax.set_yticks(range(len(stats)))
    ax.set_yticklabels(stats.index, fontsize=8)
    ax.set_title(hg, fontsize=9, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()